# DPO on Colab — Grounded RAG for Mental Health

Runs two things end-to-end on a Colab T4:
1. **`build_dpo_pairs.py`** — generate + score preference pairs (fast on CUDA vs slow on MPS).
2. **`train_dpo.py`** — TRL DPOTrainer + LoRA on the pairs.

The rest of your Mac stays untouched: this notebook clones a fresh copy of the repo, overrides device/dtype in memory for CUDA, and commits results back.

## 1. Environment setup

**Before running:** in the left sidebar, click the 🔑 icon and add a secret named `COHERE_API_KEY` with your production key.

In [ ]:
# Confirm we're on a GPU runtime. If this prints "cpu", go to Runtime → Change runtime type → T4 GPU.
import subprocess
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'NO GPU — switch runtime')

In [ ]:
# Clone the repo. Replace <YOUR_GH_USERNAME> below.
REPO_URL = 'https://github.com/<YOUR_GH_USERNAME>/Grounded-RAG-via-RL-for-mental-health.git'
!rm -rf Grounded-RAG-via-RL-for-mental-health
!git clone {REPO_URL}
%cd Grounded-RAG-via-RL-for-mental-health

In [ ]:
# Deps. TRL + PEFT for training; unsloth for the CUDA fast path.
# datasets/transformers may already be installed in Colab — the pins here won't downgrade anything critical.
!pip install -q cohere python-dotenv pyyaml transformers accelerate peft trl datasets sentencepiece
!pip install -q --no-deps unsloth  # optional; train_dpo.py falls back to plain TRL if this fails

In [ ]:
# Load Cohere key from Colab Secrets.
from google.colab import userdata
import os
os.environ['COHERE_API_KEY'] = userdata.get('COHERE_API_KEY')
assert os.environ['COHERE_API_KEY'], 'COHERE_API_KEY secret is empty — set it in the 🔑 sidebar'

## 2. Config overrides for CUDA

The repo's `configs/generation.yaml` is set to `device: mps` + `dtype: float32` for local Mac. We rewrite it for CUDA + fp16 here (only affects this Colab copy of the repo, not your Mac).

In [ ]:
import yaml, pathlib

gen_path = pathlib.Path('configs/generation.yaml')
gen = yaml.safe_load(gen_path.read_text())
gen['device'] = 'cuda'
gen['dtype'] = 'float16'   # stable on CUDA (was MPS-only issue)
gen_path.write_text(yaml.safe_dump(gen, sort_keys=False))
print(gen_path.read_text())

## 3. Build DPO pairs (smoke test → full)

Smoke test first (3 questions, ~1 min on T4). If pairs look sane, run the full 25 questions.

In [ ]:
!python -m src.scripts.build_dpo_pairs --limit 3

In [ ]:
# Eyeball one pair before committing to the full run.
import json
with open('data/dpo/pairs.jsonl') as f:
    p = json.loads(next(f))
print('=== CHOSEN (score {:.2f}) ===\n{}'.format(p['chosen_score'], p['chosen']))
print('\n=== REJECTED (score {:.2f}) ===\n{}'.format(p['rejected_score'], p['rejected']))

In [ ]:
# Full run — 25 questions, ~5-10 min on T4.
!python -m src.scripts.build_dpo_pairs

## 4. Persist the pairs back to your Mac

Pick one:
- **Option A (recommended for pairs):** commit to GitHub — small, versionable.
- **Option B:** mount Drive and copy — cleaner if you don't want a GH PAT.

For LoRA adapters (later), option B is usually better because they're 50-200MB and don't belong in git.

In [ ]:
# Option A: commit + push. Colab will prompt for GH username + PAT.
# (Skip this cell if you're going the Drive route.)
!git config user.email 'you@example.com'
!git config user.name 'you'
!git add data/dpo/
!git commit -m 'colab: build dpo pairs'
!git push

In [ ]:
# Option B: mount Drive and copy pairs there. Locally, sync via Google Drive app.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/grounded-rag-dpo
# !cp -r data/dpo /content/drive/MyDrive/grounded-rag-dpo/

## 5. Train DPO

Uses Unsloth if the pip install succeeded; falls back to plain TRL + PEFT otherwise. Both work on T4.

First run downloads Qwen2.5-1.5B (~3GB). Subsequent runs reuse the cache.

In [ ]:
!python -m src.scripts.train_dpo

In [ ]:
# Save the LoRA adapter to Drive so you don't lose it when the runtime disconnects.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/grounded-rag-dpo/checkpoints
!cp -r checkpoints/dpo /content/drive/MyDrive/grounded-rag-dpo/checkpoints/